# 🎯 Model Training Pipeline

This notebook implements a comprehensive model training process for fraud detection, incorporating various techniques to handle class imbalance and optimize model performance.

## 📋 Key Components

1. **Standard Training**
   - Baseline model without balancing or tuning
   - Helps establish performance baseline
   - Identifies potential overfitting issues

2. **Balanced Training**
   - Addresses class imbalance using undersampling
   - Improves model performance on minority class
   - Uses imbalanced-learn pipeline for proper validation

3. **Hyperparameter Optimization**
   - Grid search with cross-validation
   - Optimizes model parameters
   - Ensures robust performance across folds

## 📚 Import Libraries and Load Data

In [2]:
# Import required libraries for model training and evaluation
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV
import sys
import pickle

# Add project root to Python path for importing custom modules
sys.path.append('../../')

# Load training data and remove metadata columns
X = pd.read_parquet('../../data/train_data.parquet')
metadata_columns = ['trans_date_trans_time', 'gender', 'street']

# Separate features and target for training set
train_y = X['is_fraud']
train_X = X.drop(columns=['is_fraud'] + metadata_columns)

# Load and prepare holdout set
holdout = pd.read_parquet('../../data/holdout_data.parquet')
holdout_y = holdout['is_fraud']
holdout_X = holdout.drop(columns=['is_fraud'] + metadata_columns)

## 🎯 Standard Model Training

Training a baseline LightGBM model without any balancing or hyperparameter tuning to establish a performance baseline.

### Key Points:
- Uses default parameters
- No class balancing
- Helps identify overfitting patterns
- Serves as reference for improvements

In [3]:
# Train baseline model with default parameters
# This serves as our reference point for model improvements
score_standard_model = LGBMClassifier(objective='binary').fit(train_X, train_y)

[LightGBM] [Info] Number of positive: 6111, number of negative: 1055275
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003723 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2121
[LightGBM] [Info] Number of data points in the train set: 1061386, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005758 -> initscore=-5.151466
[LightGBM] [Info] Start training from score -5.151466


## ⚖️ Balanced Model Training

Addressing class imbalance using RandomUnderSampler to improve model performance on the minority class.

### Implementation Notes:
- Uses imbalanced-learn pipeline
- Maintains proper validation process
- Prevents data leakage
- Sampling ratio: 20% (minority/majority)

In [6]:
# Import required libraries for handling class imbalance
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

# Create pipeline with undersampling and model training
# Note: Using imbalanced-learn pipeline to prevent validation set sampling
undersample_pipe = Pipeline([
    ('sampling', RandomUnderSampler(sampling_strategy=0.2, random_state=42)),
    ('class', LGBMClassifier(objective='binary'))
])

# Train balanced model with undersampling
score_balanced_model = undersample_pipe.fit(
    train_X, train_y,
    class__eval_metric='average_precision'
)

[LightGBM] [Info] Number of positive: 6111, number of negative: 30555
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2121
[LightGBM] [Info] Number of data points in the train set: 36666, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166667 -> initscore=-1.609438
[LightGBM] [Info] Start training from score -1.609438


## 🔧 Hyperparameter Optimization

Implementing grid search with cross-validation to find optimal model parameters.

### Parameters to Optimize:
- Number of estimators: [20, 50, 200, 500]
- Number of leaves: [5, 10, 80, 100]
- Minimum data in leaf: [10, 50, 100, 200]
- Early stopping: 10 rounds

In [7]:
# Define hyperparameter grid for optimization
hyperparameters = {
    'class__n_estimators': [20, 50, 200, 500],
    "class__objective": ["binary"],
    "class__early_stopping_round": [10],
    "class__num_leaves": [5, 10, 80, 100],
    "class__min_data_in_leaf": [10, 50, 100, 200],
}

In [8]:
from imblearn.pipeline import Pipeline
#from imblearn.ensemble import BalancedBaggingClassifier

undersample_pipe = Pipeline([('sampling', RandomUnderSampler(sampling_strategy=0.1, random_state=42)) 
                             , ('class', LGBMClassifier(objective='binary'))])

score_balanced_parameter_model = GridSearchCV(undersample_pipe, param_grid=hyperparameters, cv=3, scoring='average_precision')

score_balanced_parameter_model.fit(train_X, train_y, class__eval_set=(holdout_X, holdout_y))

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] Number of positive: 4074, number of negative: 40740
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000705 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2118
[LightGBM] [Info] Number of data points in the train set: 44814, number of used features: 14
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.090909 -> initscore=-2.302585
[LightGBM] [Info] Start training from score -2.302585
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[20]	valid_0's binary_logloss: 0

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('sampling',
                                        RandomUnderSampler(random_state=42,
                                                           sampling_strategy=0.1)),
                                       ('class',
                                        LGBMClassifier(objective='binary'))]),
             param_grid={'class__early_stopping_round': [10],
                         'class__min_data_in_leaf': [10, 50, 100, 200],
                         'class__n_estimators': [20, 50, 200, 500],
                         'class__num_leaves': [5, 10, 80, 100],
                         'class__objective': ['binary']},
             scoring='average_precision')

## 💾 Save Models

In [ ]:
with open('../../models/score_standard_model.pkl', 'wb') as f:
    pickle.dump(score_standard_model, f)

with open('../../models/score_balanced_model.pkl', 'wb') as f:
    pickle.dump(score_balanced_model, f)

with open('../../models/score_balanced_parameter_model.pkl', 'wb') as f:
    pickle.dump(score_balanced_parameter_model, f)